In [4]:
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
import os


load_dotenv()

python-dotenv could not parse statement starting at line 6


True

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

def extract_clean_text(docs):
    pages = []

    for page_num, page in enumerate(docs):
        if page_num >= 20:
            text = page.page_content

            # Remove multiple spaces
            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r"\n+", " ", text)

            # Fix paragraph breaks (optional improvement)
            text = re.sub(r'\.\s+', '.\n', text)
            #text = re.sub(r"\.\n+", " ", text)
            text = re.sub(r"Fig.", " ", text)
            pages.append(
                Document(
                    page_content=text.strip(),
                    metadata=page.metadata,
                )
            )

    return pages


loader = PyMuPDFLoader(r"D:\Final_Year\data\book_3.pdf")
documents = loader.load()
documents = extract_clean_text(documents)


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
semantic_encoder = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
chunker = SemanticChunker(
    semantic_encoder,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)
documents = chunker.split_documents(documents)
documents

[Document(metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2016-01-19T01:23:56+00:00', 'source': 'D:\\Final_Year\\data\\book_3.pdf', 'file_path': 'D:\\Final_Year\\data\\book_3.pdf', 'total_pages': 385, 'format': 'PDF 1.6', 'title': 'Endoscopic Pituitary Surgery', 'author': 'Anand, Vijay K.,Schwartz, Theodore H.,American Association of Neurosurgeons.', 'subject': '', 'keywords': '', 'moddate': '2016-01-19T13:34:53+00:00', 'trapped': '', 'modDate': 'D:20160119133453Z', 'creationDate': 'D:20160119012356Z', 'page': 20}, page_content='xx Acknowledgments We would like to acknowledge the fellows and residents in the Departments of Neurosurgery and Otolaryngology at Weill Cornell Medical College/New York–Presbyterian Hos- pital for their help taking care of our patients and assistance in co-authoring our manuscripts and chapters. We would also like to acknowledge the assistance of our editors at Thieme, Kay Conerly and Lauren Henry.'),
 Document(m

In [4]:
print(len(documents))
print(type(documents))

1652
<class 'list'>


In [5]:
texts = [chunk.page_content for chunk in documents]
print(len(texts))

embeddings_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")


embeddings_text = embeddings_model.encode(texts, show_progress_bar=True)
embeddings_text


1652


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

array([[ 0.27545884, -0.3441546 , -0.3669956 , ...,  0.38380313,
         0.2390515 ,  0.1991375 ],
       [ 0.07640245, -0.6986943 ,  0.23820548, ...,  0.44894472,
         0.57463336, -0.31169558],
       [ 0.10415162, -0.31918243,  0.36163598, ...,  0.48692366,
         0.23531313, -0.38946924],
       ...,
       [-0.00891036, -0.17044038,  0.47187138, ...,  0.47025833,
         0.25301206, -0.21036947],
       [ 0.1325127 , -0.27755585,  0.14751294, ...,  0.18260005,
         0.3237785 , -0.14862612],
       [-0.6692885 ,  0.5254556 ,  0.34783003, ...,  0.09515939,
         0.48778322, -0.54615587]], shape=(1652, 768), dtype=float32)

In [6]:
print(embeddings_text.shape)
print(embeddings_model.get_sentence_embedding_dimension())

(1652, 768)
768


In [25]:
import re

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

ALLOWED_TYPES = {
    "Medication": "Medication",
    "Diagnostic_procedure": "DiagnosticProcedure",
    "Therapeutic_procedure": "TherapeuticProcedure",
    "Biological_structure": "BiologicalStructure",
    "Sign_symptom": "SignSymptom"
}

MIN_SCORE = 0.80

def validate_entities(raw_entities):
    validated = []
    seen = set()

    for e in raw_entities:
        if e["entity_group"] not in ALLOWED_TYPES:
            continue
        if e["score"] < MIN_SCORE:
            continue

        name = e["word"].strip()
        key = (name.lower(), e["entity_group"])

        if key not in seen:
            seen.add(key)
            validated.append({
                "name": name,
                "label": ALLOWED_TYPES[e["entity_group"]]
            })

    return validated

In [35]:
import json

RELATION_TYPES = [
    "DIAGNOSES",
    "TREATED_WITH",
    "DETECTS",
    "ASSOCIATED_WITH",
    "LOCATED_IN"
]


def extract_relationships_llm(chunk_text, entities, llm_client):
    entity_names = [e["word"] for e in entities]

    prompt = f"""
    Extract medical relationships from the text.

    Only use these relation types:
    {RELATION_TYPES}

    Only use these entities:
    {entity_names}

    Return output as JSON list:
    [
      {{"head": "...", "relation": "...", "tail": "..."}}
    ]

    Text:
    {chunk_text}
    """

    response = llm_client.invoke(prompt)

    try:
        relations = json.loads(response)
    except:
        relations = []

    # Validate triples
    validated = []
    for r in relations:
        if (
            r["relation"] in RELATION_TYPES and
            r["head"] in entity_names and
            r["tail"] in entity_names
        ):
            validated.append(r)

    return validated


In [36]:
def create_chunk(tx, chunk_id, text, embedding):
    query = """
    MERGE (c:Chunk {id: $id})
    SET c.text = $text,
        c.embedding = $embedding
    """
    tx.run(query, id=chunk_id, text=text, embedding=embedding)


def create_entity_and_link(tx, chunk_id, label, name):
    query = f"""
    MATCH (c:Chunk {{id: $chunk_id}})
    MERGE (e:{label} {{name: $name}})
    MERGE (c)-[:MENTIONS]->(e)
    """
    tx.run(query, chunk_id=chunk_id, name=name)

def create_entity_relationship(tx, head_label, head_name, relation, tail_label, tail_name):
    query = f"""
    MATCH (h:{head_label} {{name: $head_name}})
    MATCH (t:{tail_label} {{name: $tail_name}})
    MERGE (h)-[:{relation}]->(t)
    """
    tx.run(query,
           head_name=head_name,
           tail_name=tail_name)

def link_chunks(tx, prev_id, current_id):
    query = """
    MATCH (a:Chunk {id: $prev})
    MATCH (b:Chunk {id: $current})
    MERGE (a)-[:NEXT]->(b)
    """
    tx.run(query, prev=prev_id, current=current_id)


In [37]:
def ingest_chunks(documents, embeddings, ner_pipeline, llm_client, driver):
    previous_chunk_id = None

    for i, (doc, embed) in enumerate(zip(documents, embeddings)):

        chunk_id = i
        text = clean_text(doc.page_content)

        # Make sure embedding is list
        embedding = embed.tolist() if hasattr(embed, "tolist") else embed

        # -------------------------
        # STEP 1: NER Extraction
        # -------------------------
        raw_entities = ner_pipeline(text)

        # -------------------------
        # STEP 2: Deduplicate
        # -------------------------
        raw_entities = deduplicate_entities(raw_entities)

        # -------------------------
        # STEP 3: Validate & Normalize
        # -------------------------
        entities = validate_entities(raw_entities)

        with driver.session() as session:

            # Always store chunk
            session.execute_write(create_chunk, chunk_id, text, embedding)

            # Sequential linking
            if previous_chunk_id is not None:
                session.execute_write(link_chunks, previous_chunk_id, chunk_id)

            if entities:
                # Store entities + link to chunk
                for e in entities:
                    session.execute_write(
                        create_entity_and_link,
                        chunk_id,
                        e["entity_group"],  # label
                        e["word"]           # name
                    )

                # -------------------------
                # STEP 4: LLM Relationship Extraction
                # -------------------------
                relations = extract_relationships_llm(text, entities, llm_client)

                for r in relations:
                    head_label = next(e["entity_group"] for e in entities if e["word"] == r["head"])
                    tail_label = next(e["entity_group"] for e in entities if e["word"] == r["tail"])

                    session.execute_write(
                        create_entity_relationship,
                        head_label,
                        r["head"],
                        r["relation"],
                        tail_label,
                        r["tail"]
                    )

        previous_chunk_id = chunk_id

    print("Neo4j Hybrid Ingestion Complete")

In [53]:
from neo4j import GraphDatabase
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME =  os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

try:
    driver.verify_connectivity()
    print("Connection established successfully.")
except Exception as e:
    print(f"Connection failed: {e}")

Connection established successfully.


In [43]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


llm_client = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [44]:
def deduplicate_entities(entities):
    seen = set()
    unique = []

    for e in entities:
        key = (e["word"].lower(), e["entity_group"])
        if key not in seen:
            seen.add(key)
            unique.append(e)

    return unique
import re

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text
def remove_noise(entity):
    word = entity["word"].strip()

    # Remove punctuation or partial tokens
    if len(word) <= 2:
        return False

    # Remove numeric-only entities
    if word.isdigit():
        return False

    # Remove entities containing only numbers and symbols
    if re.fullmatch(r"[\d\W]+", word):
        return False

    return True
ALLOWED_TYPES = {
    "Disease_disorder",
    "Medication",
    "Diagnostic_procedure",
    "Therapeutic_procedure",
    "Biological_structure",
    "Sign_symptom"
}

def check_allowed_type(entity):
    return entity["entity_group"] in ALLOWED_TYPES
TYPE_SCORE_THRESHOLD = {
    "Disease_disorder": 0.85,
    "Medication": 0.75,
    "Diagnostic_procedure": 0.80,
    "Therapeutic_procedure": 0.80,
    "Biological_structure": 0.85,
    "Sign_symptom": 0.85
}

def check_score(entity):
    threshold = TYPE_SCORE_THRESHOLD.get(entity["entity_group"], 0.85)
    return entity["score"] >= threshold
FORCE_TYPE_MAP = {
    "ipss": "Diagnostic_procedure",
    "acth": "Hormone",
    "crh": "Hormone"
}

def override_type(entity):
    word = entity["word"].lower()

    if word in FORCE_TYPE_MAP:
        entity["entity_group"] = FORCE_TYPE_MAP[word]

    return entity
def canonicalize(entity):
    word = entity["word"].strip()

    if word.isupper():
        return word  # keep acronyms

    return word.lower()
def validate_entities(entities):
    validated = []

    for e in entities:
        if not remove_noise(e):
            continue

        if not check_allowed_type(e):
            continue

        if not check_score(e):
            continue

        e = override_type(e)
        e["word"] = canonicalize(e)

        validated.append(e)

    return validated


In [45]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("d4data/biomedical-ner-all")
model = AutoModelForTokenClassification.from_pretrained("d4data/biomedical-ner-all")

ner_pipeline = pipeline(task="ner",model=model,tokenizer=tokenizer,aggregation_strategy="simple")



Device set to use cpu


In [46]:
ingest_chunks(
    documents=documents,
    embeddings=embeddings_text,
    ner_pipeline=ner_pipeline,
    llm_client=llm_client,
    driver=driver
)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 56.039517186s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_d

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 53.674108621s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 53
}
]

In [69]:
from langchain_neo4j import GraphCypherQAChain
from langchain_neo4j import Neo4jGraph
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()


graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database= os.getenv("NEO4J_DATABASE")
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # or mixtral-8x7b-32768
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

schema_info = """
Labels: Chunk, Medication, SignSymptom, BiologicalStructure, DiagnosticProcedure, TherapeuticProcedure
Properties:
  Chunk.text
  Medication.name
  SignSymptom.name
  BiologicalStructure.name
  DiagnosticProcedure.name
  TherapeuticProcedure.name
Relationships:
  Chunk-MENTIONS->Entity
  Entity-[DIAGNOSES/TREATED_WITH/DETECTS/ASSOCIATED_WITH/LOCATED_IN]->Entity
"""



chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    schema_info=schema_info)
query = "what is Anatomy of the Pituitary Gland and Parasellar Region"
result = chain.invoke(query)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Chunk)-[:MENTIONS]->(bs:Biological_structure {name: "Pituitary Gland"}) 
OPTIONAL MATCH (c)-[:MENTIONS]->(ps:Biological_structure {name: "Parasellar Region"}) 
RETURN c.text, bs.name, ps.name
Full Context:
[]

> Finished chain.


In [70]:
print(result)

{'query': 'what is Anatomy of the Pituitary Gland and Parasellar Region', 'result': "I don't know the answer."}


In [75]:
query = "Anatomy of the Pituitary Gland and Parasellar Region"
query_embedding = embeddings_model.encode(query).tolist()  # convert to list for Neo4j


top_k = 5  # number of chunks you want

with driver.session() as session:
    result = session.run(
        """
        MATCH (c:Chunk)
        WITH c, gds.similarity.cosine(c.embedding, $query_embedding) AS score
        RETURN c.id AS chunk_id, c.text AS text, score
        ORDER BY score DESC
        LIMIT $top_k
        """,
        query_embedding=query_embedding,
        top_k=top_k
    )
    top_chunks = [r for r in result]
    for r in top_chunks:
        print(r["chunk_id"], r["score"], r["text"][:150])
chunk_ids = [r["chunk_id"] for r in top_chunks]

with driver.session() as session:
    result = session.run(
        """
        MATCH (c:Chunk)-[:MENTIONS]->(e)
        WHERE c.id IN $chunk_ids
        RETURN c.id AS chunk_id, c.text AS chunk_text, e.name AS entity, labels(e) AS label
        """,
        chunk_ids=chunk_ids
    )
    for r in result:
        print(r["chunk_id"], r["entity"], r["label"])


context = "\n".join([r["text"] for r in top_chunks])
question = "What is the anatomy of the Pituitary Gland?"

prompt = f"""
You are a medical assistant.
Use the following context to answer the question:

CONTEXT:
{context}

QUESTION:
{question}
"""

answer = llm.invoke(prompt)
print(answer)

15 0.6199142521366221 Endoscopic Pituitary Surgery 2 1.2 Diagram demonstrating the lateral rhinotomy exposure of the septum and nasal turbinates of the nose just prior to e
23 0.5928691642895986 1 History of Pituitary Surgery  1.9 Illustration depicting selective removal of a pituitary microadenoma while preserving pituitary function. (From H
20 0.5706657455253874 Endoscopic Pituitary Surgery  1.7 Illustrations depicting Hardy’s sublabial transseptal transsphe- noidal approach. (A) Sublabial incision. (B) Eleva
21 0.5415527049527135 (From Jho HD, et al. Endoscopic pituitary surgery: an early experience. Surg Neurol 1997;47:213–222. Reprinted with permission.) and parasellar lesion
29 0.5408845110679373 Artico M, Pastore FS, Fraioli B, Giuffrè R. The contribution of Davide Giordano (1864–1954) to pituitary surgery: the transglabellar-nasal approach. 
15 rhino ['Therapeutic_procedure']
15 pituitary surgery ['Therapeutic_procedure']
15 septum ['Biological_structure']
15 nasal turbinate

In [80]:
print(answer.content)

As a medical assistant, I must clarify that the provided context does not directly describe the anatomy of the pituitary gland. However, I can provide a general overview of the pituitary gland's anatomy based on my knowledge.

The pituitary gland, also known as the hypophysis, is a small endocrine gland located at the base of the brain, in a small bony cavity called the sella turcica. It is situated near the center of the brain, above the roof of the mouth and behind the bridge of the nose.

The pituitary gland has two main parts: the anterior pituitary (adenohypophysis) and the posterior pituitary (neurohypophysis). The anterior pituitary produces and secretes several hormones that regulate various bodily functions, such as growth, metabolism, and reproductive processes. The posterior pituitary stores and releases hormones produced by the hypothalamus, a nearby region of the brain.

The pituitary gland is surrounded by several important structures, including:

1. The sphenoid sinus: a

In [81]:
driver.close()